In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import statistics

from main import load_graph_and_communities, run_configured_sicp
from removal import PHG_community_influence, remove_nodes, inverse_graph, inverse_communities


p = 0.2

temp_graph, communities = load_graph_and_communities("Geometry", tau=2, verbose=True)

all_nodes = set.union(*temp_graph.values())
N_ = len(all_nodes)
K = int(p * float(N_))

rem = PHG_community_influence(temp_graph, communities, K)

print(f"final measure: {rem}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load intermediate results
df = pd.read_csv("outputs/multithreaded_intermediate_runs_geometry.csv")

# Get unique p values and strategies
p_values = sorted(df['p'].unique())
strategies = sorted(df['strategy'].unique())

# Create subplots - one for each p value
fig, axes = plt.subplots(1, len(p_values), figsize=(16, 5), sharey=True)
if len(p_values) == 1:
    axes = [axes]

# Color palette for strategies
colors = plt.cm.tab10(np.linspace(0, 1, len(strategies)))
color_map = {strategy: colors[i] for i, strategy in enumerate(strategies)}

# Add traces for each strategy and p value
for col_idx, p in enumerate(p_values):
    ax = axes[col_idx]
    p_data = df[df['p'] == p]
    
    for strategy in strategies:
        strategy_data = p_data[p_data['strategy'] == strategy].sort_values('timestep')
        if len(strategy_data) > 0:
            ax.plot(
                strategy_data['timestep'],
                strategy_data['mean_infected_pct'],
                marker='o',
                label=strategy,
                linewidth=2,
                color=color_map[strategy],
                markersize=4
            )
    
    ax.set_xlabel('Timestep')
    ax.set_title(f'p = {p:.2f}')
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Mean Infected %')

# Add legend to the rightmost plot
handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1, 0.5), ncol=1)

plt.tight_layout()
plt.subplots_adjust(right=0.85)
plt.suptitle('SICP Infection Prevalence by Strategy and Removal Fraction', y=1.02)
plt.show()

In [ ]:
from ept import *
from main import *

temp_graph, communities = load_graph_and_communities("Geometry", tau=2, verbose=True)

ept = build_ept(temp_graph, beta = 0.2)


In [ ]:
from removal import (
    ept_out_strength,
    random_based_removal,
    hyperdegree_based_removal,
    degree_based_removal,
    remove_nodes
)

# Parameters
BETA = 0.3
T = 25
RUNS = 10
rng_seed = 175
p = 0.05
seed_iterations = 50

# Prepare graph and calculate K
all_nodes = set.union(*temp_graph.values())
N_ = len(all_nodes)
K = int(p * float(N_))

print(f"Running removal strategies with p={p}, K={K} nodes to remove")
print(f"Graph has {len(temp_graph)} hyperedges and {N_} nodes")

# Apply removal strategies
graph_degree = remove_nodes(temp_graph, degree_based_removal(temp_graph, K))
graph_hyperdegree = remove_nodes(temp_graph, hyperdegree_based_removal(temp_graph, K))
graph_random = remove_nodes(temp_graph, random_based_removal(temp_graph, K))
graph_ept = remove_nodes(temp_graph, ept_out_strength(temp_graph, BETA, K))

print("Running SICP for each strategy with intermediate timesteps...")

# Run SICP for each strategy and collect intermediate results
prevalence_degree = run_configured_sicp_intermediates(graph_degree, seed_iterations, beta=BETA, T=T, runs=RUNS, rng_seed=rng_seed)
prevalence_hyperdegree = run_configured_sicp_intermediates(graph_hyperdegree, seed_iterations, beta=BETA, T=T, runs=RUNS, rng_seed=rng_seed)
prevalence_random = run_configured_sicp_intermediates(graph_random, seed_iterations, beta=BETA, T=T, runs=RUNS, rng_seed=rng_seed)
prevalence_ept = run_configured_sicp_intermediates(graph_ept, seed_iterations, beta=BETA, T=T, runs=RUNS, rng_seed=rng_seed)

# Plot comparison over time
fig, ax = plt.subplots(figsize=(10, 6))

timesteps = list(range(T + 1))
strategies = {
    'Degree': prevalence_degree,
    'Hyperdegree': prevalence_hyperdegree,
    'Random': prevalence_random,
    'EPT Out-Strength': prevalence_ept
}

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
color_map = {strategy: colors[i] for i, strategy in enumerate(strategies.keys())}

for strategy, prevalence in strategies.items():
    ax.plot(
        timesteps,
        prevalence,
        marker='o',
        label=strategy,
        linewidth=2,
        color=color_map[strategy],
        markersize=4
    )

ax.set_xlabel('Timestep', fontsize=12)
ax.set_ylabel('Mean Infected Nodes', fontsize=12)
ax.set_title(f'SICP Infection Prevalence by Removal Strategy (p={p}, K={K})', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left', fontsize=11)

plt.tight_layout()
plt.show()

print(f"\nFinal infected counts:")
for strategy, prevalence in strategies.items():
    print(f"  {strategy}: {prevalence[-1]:.1f} nodes")